# Tunnelling in a double well

*quantum · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/double_well)

A particle in a symmetric double-well potential. torch.linalg.eigh shows that the two lowest states form a close pair: an even state and an odd state, split by a tiny tunnelling energy $\Delta E$. Their sum is a particle sitting in the left well; as the two relative phases drift apart it tunnels to the right well and back with period $2\pi/\Delta E$, although its energy is below the barrier.

**Vector form**

$$\hat H = -\tfrac12\,\frac{d^2}{dx^2} + V_0\Big(\frac{x^2}{a^2} - 1\Big)^2,\qquad \psi(0) = \frac{\varphi_0 + \varphi_1}{\sqrt2},\qquad P_L(t) = \int_{-\infty}^0 |\psi|^2 dx \approx \cos^2\frac{\Delta E\,t}{2}$$

**Index (tensor) form**

$$H_{xy}\,\varphi_{yn} = E_n\,\varphi_{xn},\qquad \psi_x(t) = \varphi_{xn}\,c_n\,e^{-iE_n t},\qquad P_L(t) = \psi^*_x\,\Theta_x\,\psi_x\,dx$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $V_x$ | `double_well(x, V0, a)` | (N,) | quartic potential: minima at $x = \pm a$, barrier $V_0$ at $x = 0$ |
| $H_{xy}$ | `hamiltonian(x, dx, V)` | (N, N) | kinetic $-\tfrac12\,D2$ plus $V$ on the diagonal |
| $E_n,\ \varphi_{xn}$ | `torch.linalg.eigh(H)` | (N,), (N, N) | energies and states; $\varphi_0$ even, $\varphi_1$ odd, $\Delta E = E_1 - E_0$ |
| $c_n$ | `left_state(N)` | (N,) complex | $(1, 1, 0, 0, \dots)/\sqrt2$: equal mix of the doublet, localised on the left |
| $\psi_x(t)$ | `einsum('xn,n->x', phi, c * exp(-1j*E*t))` | (N,) complex | repeated $n$ ⇒ summed: exact evolution in the eigenbasis |
| $\Theta_x$ | `(x < 0)` | (N,) | step function selecting the left half; $P_L$ sums over $x$ |

Units $\hbar = m = 1$. The doublet energies lie below the barrier top $V_0$, so classically the particle could never cross. With only two states in play, $\psi(t) \propto \varphi_0 + e^{-i\Delta E\,t}\varphi_1$ (up to a global phase): it is $(\varphi_0 + \varphi_1)$ on the left at $t = 0$ and $(\varphi_0 - \varphi_1)$ on the right at $t = \pi/\Delta E$. $P_L(t) = \cos^2(\Delta E\,t/2)$ holds up to the small overlap of each state with the "wrong" well. $\Delta E$ shrinks roughly exponentially with the barrier height and width (a WKB factor $e^{-\int\kappa\,dx}$), so a slightly higher barrier slows tunnelling enormously — the simulated time is measured in tunnelling periods $2\pi/\Delta E$ so the animation always shows whole oscillations.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/double_well/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import left_probability, two_state_prediction


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    V0, a, n = params["V0"], params["a"], params["n_grid"]
    x, dx = tensors.grid(n, 4 * a + 6, dev)  # box reaches well past both minima
    V = tensors.double_well(x, V0, a)
    H = tensors.hamiltonian(x, dx, V)

    energies, phi = solve.eigenstates(H, x, dx)
    period = solve.tunnelling_period(energies)
    times, psi = solve.solve(phi, energies, tensors.left_state(n, dev), params["periods"] * period)
    p_left = left_probability(psi, x, dx)
    delta_E = (energies[1] - energies[0]).item()

    print(f"E_0 = {energies[0]:.5f}   E_1 = {energies[1]:.5f}   barrier V0 = {V0:.3f}")
    print(f"splitting dE = {delta_E:.5f}   tunnelling period 2pi/dE = {period:.2f}")
    print(f"P_left: min {p_left.min():.4f}  (two-state cos^2 prediction: 0)")
    prediction = two_state_prediction(times, delta_E)
    return plot.animate(x, times, psi, V, phi, energies, p_left, prediction, out_dir / "double_well.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/double_well/equation.py
r"""Physics → tensor translation.

Time-independent Schrödinger equation in a symmetric double well (units ħ = m = 1):

    \hat H \varphi = -\tfrac12 \varphi'' + V(x) \varphi = E \varphi,      V(x) = V_0 (x^2/a^2 - 1)^2.

On a grid, with the eigenvectors as the columns of \varphi_{xn}:

    H_{xy} = -\tfrac12 D2_{xy} + V_x \delta_{xy},      H_{xy} \varphi_{yn} = E_n \varphi_{xn}      (y summed)

Start in c = (1, 1, 0, …)/\sqrt2, i.e. \psi(0) = (\varphi_0 + \varphi_1)/\sqrt2, and evolve exactly:

    \psi_x(t) = \varphi_{xn} c_n e^{-i E_n t}                   (n summed)

The probability in the left well uses the step function \Theta_x = 1 for x < 0:

    P_L(t) = \psi^*_x \Theta_x \psi_x \, dx                       (x summed)

For two states this gives P_L \approx \tfrac12 (1 + \cos \Delta E t) = \cos^2(\Delta E t / 2), \Delta E = E_1 - E_0.
"""

import torch


def evolve(phi: torch.Tensor, energies: torch.Tensor, c: torch.Tensor, t: float) -> torch.Tensor:
    r"""\psi_x(t) = \varphi_{xn} c_n e^{-i E_n t}.   phi: (N, M), energies: (M,), c: (M,) → (N,) complex"""
    phase = torch.exp(-1j * energies * t)  # (M,)  e^{-i E_n t}
    return torch.einsum("xn,n->x", phi.to(c.dtype), c * phase)


def left_probability(psi: torch.Tensor, x: torch.Tensor, dx: float) -> torch.Tensor:
    r"""P_L = \psi^*_x \Theta_x \psi_x dx  (x summed).   psi: (..., N), x: (N,) → (...)"""
    theta = (x < 0).to(psi.dtype)  # (N,) step function Θ_x
    return torch.einsum("...x,x,...x->...", psi.conj(), theta, psi).real * dx


def two_state_prediction(times: torch.Tensor, delta_E: float) -> torch.Tensor:
    r"""P_L(t) = \cos^2(\Delta E\, t / 2) for an ideal two-level system.   (T,) → (T,)"""
    return torch.cos(delta_E * times / 2) ** 2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/double_well/tensors.py
"""Build the concrete tensors: the grid, the double-well potential, the Hamiltonian and the start coefficients."""

import math

import torch


def grid(n: int, L: float, device: torch.device) -> tuple[torch.Tensor, float]:
    """N interior points of the box [−L/2, L/2], symmetric about x = 0 (ψ = 0 at the walls). Returns (x (N,), dx)."""
    dx = L / (n + 1)
    x = -L / 2 + dx * torch.arange(1, n + 1, dtype=torch.float64, device=device)  # (N,)
    return x, dx


def double_well(x: torch.Tensor, V0: float, a: float) -> torch.Tensor:
    """V_x = V0 (x²/a² − 1)²: zero at the minima x = ±a, height V0 at x = 0. Shape (N,)."""
    return V0 * (x**2 / a**2 - 1) ** 2


def second_derivative(n: int, dx: float, device: torch.device) -> torch.Tensor:
    """D2_{xy}: −2 on the diagonal, 1 on the two neighbouring diagonals, all over dx². Shape (N, N)."""
    main = -2.0 * torch.ones(n, dtype=torch.float64, device=device)  # (N,)
    side = torch.ones(n - 1, dtype=torch.float64, device=device)  # (N-1,)
    return (torch.diag(main) + torch.diag(side, 1) + torch.diag(side, -1)) / dx**2


def hamiltonian(x: torch.Tensor, dx: float, V: torch.Tensor) -> torch.Tensor:
    """H_{xy} = −½ D2_{xy} + V_x δ_{xy}. Shape (N, N), symmetric."""
    return -0.5 * second_derivative(len(x), dx, x.device) + torch.diag(V)


def left_state(n: int, device: torch.device) -> torch.Tensor:
    """c_n = (1, 1, 0, …, 0)/√2: equal mix of the even and odd ground doublet. Shape (N,) complex."""
    c = torch.zeros(n, dtype=torch.complex128, device=device)
    c[:2] = 1 / math.sqrt(2)
    return c


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/double_well/solve.py
"""Diagonalise H once, fix the signs of the doublet, then evolve exactly in the eigenbasis."""

import math

import torch

from .equation import evolve


def eigenstates(H: torch.Tensor, x: torch.Tensor, dx: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (E_n (N,), φ_{xn} (N, N)) with ∫|φ|² dx = 1.

    eigh may return any eigenvector with either sign. We choose every state to be positive in the
    left well (summed over x < 0), so that φ_0 + φ_1 is the left-localised combination.
    """
    energies, phi = torch.linalg.eigh(H)
    phi = phi / math.sqrt(dx)
    left_weight = phi[x < 0].sum(dim=0)  # (N,)
    return energies, phi * torch.where(left_weight < 0, -1.0, 1.0)


def tunnelling_period(energies: torch.Tensor) -> float:
    """T = 2π / ΔE with ΔE = E_1 − E_0: time to tunnel to the right well and back."""
    return 2 * math.pi / (energies[1] - energies[0]).item()


def solve(
    phi: torch.Tensor, energies: torch.Tensor, c: torch.Tensor, t_max: float, n_steps: int = 240
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), ψ (T, N) complex). No time stepping: every t is evaluated exactly."""
    times = torch.linspace(0.0, t_max, n_steps + 1, dtype=torch.float64)
    psi = torch.stack([evolve(phi, energies, c, t) for t in times.tolist()])  # (T, N)
    return times, psi


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/double_well/plot.py
"""Animate |ψ|² tunnelling between the wells (left) and the probability P_L(t) in the left well (right)."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    x: torch.Tensor,
    times: torch.Tensor,
    psi: torch.Tensor,
    V: torch.Tensor,
    phi: torch.Tensor,
    energies: torch.Tensor,
    p_left: torch.Tensor,
    prediction: torch.Tensor,
    path: Path,
) -> Path:
    xs, Vs = x.cpu().numpy(), V.cpu().numpy()
    prob = (psi.abs() ** 2).cpu().numpy()  # (T, N)
    t = times.cpu().numpy()
    pl, pred = p_left.cpu().numpy(), prediction.cpu().numpy()
    E = energies[:2].cpu().numpy()
    doublet = phi[:, :2].T.cpu().numpy()  # (2, N)  φ_0 (even) and φ_1 (odd)
    V0 = float(V[len(V) // 2])  # barrier height at x ≈ 0

    fig, (ax, ax_p) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1.5, 1]})
    view = xs[Vs < 2.2 * V0]  # region where the potential is below twice the barrier
    ax.set_xlim(view.min(), view.max())
    ax.set_ylim(0, 2.0 * V0)
    ax.set_xlabel("x")
    ax.set_ylabel("V(x),  E_n")
    ax.plot(xs, Vs, color="#6b7280", lw=2)
    scale = 0.12 * V0 / abs(doublet).max()
    for En, state in zip(E, doublet):  # faint doublet φ_0, φ_1 drawn at their energies
        ax.plot(xs, En + scale * state, color="#9ca3af", lw=0.8, ls="--")

    ax_psi = ax.twinx()
    ax_psi.grid(False)
    ax_psi.set_ylim(0, 2.2 * prob.max())
    ax_psi.set_yticks([])
    (line,) = ax_psi.plot([], [], color=ACCENT[0], lw=2)
    fill = [ax_psi.fill_between(xs, 0 * xs, color=ACCENT[0], alpha=0.25)]
    clock = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top", family="monospace")
    ax.set_title("|ψ|² (blue),  doublet φ0, φ1 (dashed)", fontsize=10)

    ax_p.set_xlim(t[0], t[-1])
    ax_p.set_ylim(-0.05, 1.05)
    ax_p.set_xlabel("t")
    ax_p.set_ylabel("probability in left well")
    ax_p.plot(t, pred, color="#9ca3af", ls="--", lw=1, label="cos²(ΔE t/2)")
    (curve,) = ax_p.plot([], [], color=ACCENT[1], lw=2, label="P_L(t)")
    (dot,) = ax_p.plot([], [], "o", color=ACCENT[1])
    ax_p.legend(loc="lower left", fontsize=8)

    def update(step: int):
        line.set_data(xs, prob[step])
        fill[0].remove()
        fill[0] = ax_psi.fill_between(xs, prob[step], color=ACCENT[0], alpha=0.25)
        curve.set_data(t[: step + 1], pl[: step + 1])
        dot.set_data([t[step]], [pl[step]])
        clock.set_text(f"t = {t[step]:6.1f}\nP_L = {pl[step]:.3f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'V0': 3.0,
    'a': 1.5,
    'periods': 2.0,
    'n_grid': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.double_well")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.double_well.main")
problem_dir = Path("problems/quantum/double_well")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
